# 01. Pogoda — ERA5-Land

Jedna doba (3.02.2012, najzimniej) dla całej Polski. Wymaga konta CDS i pliku `~/.cdsapirc` (README, „Dostępy”).
`pip install "cdsapi>=0.7.7" xarray netCDF4 matplotlib`

In [ ]:
import cdsapi
import xarray as xr
import matplotlib.pyplot as plt

# ERA5-Land: 1 doba, cala Polska
cdsapi.Client().retrieve("reanalysis-era5-land", {
    "variable": ["2m_temperature", "skin_temperature", "snow_depth", "total_precipitation",
                 "soil_temperature_level_1", "volumetric_soil_water_layer_1"],
    "year": "2012", "month": "02", "day": "03",
    "time": [f"{h:02d}:00" for h in range(24)],
    "area": [55.0, 14.0, 49.0, 24.2],
    "data_format": "netcdf",
    "download_format": "unarchived",
}, "era5land_sample.nc")

In [ ]:
ds = xr.open_dataset("era5land_sample.nc")
ds

Nazwy, jednostki, wymiary:

In [ ]:
for name, var in ds.data_vars.items():
    print(name, "|", var.attrs.get("units"), "|", var.attrs.get("long_name"), "|", dict(var.sizes))

Temperatura jest w kelwinach, śnieg w metrach. Nad morzem i za granicą lądu są `NaN`.

In [ ]:
time = "valid_time" if "valid_time" in ds.dims else "time"
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
(ds["t2m"].min(time) - 273.15).plot(ax=ax[0], cmap="Blues_r")
(ds["sde"].mean(time) * 100).plot(ax=ax[1], cmap="PuBu")
ax[0].set_title("min temperatura [°C]")
ax[1].set_title("śnieg [cm]")
plt.show()

Jeden punkt (Warszawa). Opad `tp` **narasta** przez dobę — to nie jest opad z jednej godziny.

In [ ]:
point = ds.sel(latitude=52.2, longitude=21.0, method="nearest").to_dataframe()
point[["t2m", "skt", "stl1"]] = point[["t2m", "skt", "stl1"]] - 273.15
point[["t2m", "skt", "stl1", "sde", "tp"]].round(4)